In [22]:
import matplotlib.pyplot as plt
import anndata
import scanpy as sc
import snapatac2 as snap
import numpy as np
import pandas as pd
import os
import scanpy.external as sce
import seaborn as sns
from sklearn.metrics import silhouette_score
import numpy as np
from scipy.stats import chi2

import warnings
warnings.filterwarnings("ignore")
import glob


In [23]:
files = glob.glob('/data1st2/junyi/output/atac1112/subset/region_nt/*sc_subset.h5ad')
for file in files:
    # adata = sc.read_h5ad(file)
    # sc.pp.calculate_qc_metrics(adata, inplace=True)
    # sc.pl.umap(adata, color=['total_counts', 'n_genes_by_counts','sample'], wspace=0.4)
    # sc.pl.violin(adata, ['total_counts', 'n_genes_by_counts'],jitter=False, groupby='sample', rotation=90)    
    adata = sc.read_h5ad(file)
    ad_bulk = sc.get.aggregate(adata,by='celltype.L2.condition',func='sum',layer="counts")
    ad_bulk.X = ad_bulk.layers['sum'].astype(int)
    sc.pp.normalize_total(ad_bulk, target_sum=1e6)
    sc.pp.log1p(ad_bulk)
    ad_bulk.write_h5ad(f'/data1st2/junyi/output/atac1112/subset/region_nt/{os.path.basename(file).replace("_sc_subset.h5ad", "_sc_bulk.h5ad")}')

In [9]:
for mod in ['5mC', '5hmC']:
    df_dmr_5mc = df_dmr[df_dmr['mod']==mod]
    df_dmr_5mc['Region'] = df_dmr_5mc['comparision'].str[3:6]
    df_dmr_5mc = df_dmr_5mc#[df_dmr_5mc['Region']=='AMY']
    df_dmr_5mc_expanded = df_dmr_5mc.dmr.str.split('[|_:-]', expand=True)
    df_dmr_5mc_expanded.columns = ['methtype', 'motif2', 'chr','start', 'end']
    # Expand DMRs with length <100 to 100bp centered at original center
    df_dmr_5mc_expanded['length'] = df_dmr_5mc_expanded['end'].astype(int) - df_dmr_5mc_expanded['start'].astype(int)
    df_dmr_5mc_expanded['center'] = (df_dmr_5mc_expanded['end'].astype(int) + df_dmr_5mc_expanded['start'].astype(int)) //2
    df_dmr_5mc_expanded['start_expanded'] = df_dmr_5mc_expanded['center'] - 50
    df_dmr_5mc_expanded['end_expanded'] = df_dmr_5mc_expanded['center'] + 50
    # if legthn <100, expand start = start_expanded, end = end_expanded
    df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'start'] = df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'start_expanded']
    df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'end'] = df_dmr_5mc_expanded.loc[df_dmr_5mc_expanded['length']<100, 'end_expanded']

    df_dmr_5mc =  pd.concat([df_dmr_5mc, df_dmr_5mc_expanded], axis=1)
    df_dmr_5mc.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/{mod}_annotation.csv')
    df_bed= df_dmr_5mc.loc[:,['chr', 'start', 'end']].drop_duplicates()
    df_bed['chr'] = 'chr' + df_bed['chr'].astype(str)
    df_bed.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/dmr_{mod}.bed', sep='\t', header=False, index=False)


In [47]:
df_bed.drop_duplicates(['chr', 'start', 'end'])

,chr,start,end
FC-AMY_vs_FW-AMY:methylDMR.1,chr1,100000185,100000671
FC-AMY_vs_FW-AMY:methylDMR.2,chr1,100213290,100213390
FC-AMY_vs_FW-AMY:methylDMR.3,chr1,100224782,100224882
FC-AMY_vs_FW-AMY:methylDMR.4,chr1,100226519,100226652
FC-AMY_vs_FW-AMY:methylDMR.5,chr1,100269082,100269242
...,...,...,...
297040,chrX,170011131,170011231
297052,chrX,170018649,170018749
297063,chrX,170019104,170019204
297083,chrY,90785820,90785920


In [10]:
adata_concat = snap.read_dataset('/data2st1/junyi/output/atac0627/doublet_filtered.h5ads/_dataset.h5ads')

In [11]:
%time hm5c_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_5hmC.bed')
hm5c_mat.write(f"output/atac1112/3REGIONS_5hmc_new.h5ads")

... storing 'sample' as categorical
... storing 'celltype.L1' as categorical
... storing 'celltype.L2' as categorical
... storing 'Neurotransmitter_celltype' as categorical
... storing 'celltype.L1_ct' as categorical
... storing 'Sample_name' as categorical
... storing 'Condition' as categorical
... storing 'Region' as categorical
... storing 'celltype.L2.raw' as categorical
... storing 'region_nt' as categorical
... storing 'celltype.L3' as categorical
... storing 'celltype.L4' as categorical
... storing 'celltype.L2.refined' as categorical


CPU times: user 1h 57min, sys: 24min 12s, total: 2h 21min 13s
Wall time: 6min 14s


In [12]:
%time m5c_mat = snap.pp.make_peak_matrix(adata_concat,peak_file='/data2st1/junyi/output/atac1112/cCRE/dmr_5mC.bed')
m5c_mat.write(f"output/atac1112/3REGIONS_5mc_new.h5ads")
adata_concat.close()

... storing 'sample' as categorical
... storing 'celltype.L1' as categorical
... storing 'celltype.L2' as categorical
... storing 'Neurotransmitter_celltype' as categorical
... storing 'celltype.L1_ct' as categorical
... storing 'Sample_name' as categorical
... storing 'Condition' as categorical
... storing 'Region' as categorical
... storing 'celltype.L2.raw' as categorical
... storing 'region_nt' as categorical
... storing 'celltype.L3' as categorical
... storing 'celltype.L4' as categorical
... storing 'celltype.L2.refined' as categorical


CPU times: user 1h 57min 15s, sys: 23min 41s, total: 2h 20min 56s
Wall time: 6min 18s


In [ ]:
annodict= {}
for methtype in ['5mC', '5hmC']:
    if methtype == '5mC':
        annodict[methtype] = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/5mC_annotation.csv',index_col=0)
    else:
        annodict[methtype] = pd.read_csv('/data2st1/junyi/output/atac1112/cCRE/5hmC_annotation.csv',index_col=0)
    for region in ['AMY', 'HIP', 'PFC']:
        df_region = annodict[methtype][annodict[methtype]['Region']==region]
        df_region.loc[df_region['length']<100, 'start'] = df_region.loc[df_region['length']<100, 'start_expanded']
        df_region.loc[df_region['length']<100, 'end'] = df_region.loc[df_region['length']<100, 'end_expanded']
        df_bed = df_region.loc[:,['chr', 'start', 'end']].drop_duplicates()
        df_bed['chr'] = 'chr' + df_bed['chr'].astype(str)
        df_bed.to_csv(f'/data2st1/junyi/output/atac1112/cCRE/{methtype}_{region}.bed', sep='\t', header=False, index=False)

In [6]:
df_region

,mod,motif,dmr,ifdifferent,score,num_sites,effect_size,case1_sig,case2_sig,diff.Methy,...,Region,methtype,motif2,chr,start,end,length,center,start_expanded,end_expanded
FC-PFC_vs_FW-PFC:methylDMR.1,5hmC,CG,5hmC|CG_1:100024090-100024167,different,-24.213754,6,-0.235359,0.058975,0.294334,-0.235359,...,PFC,5hmC,CG,1,100024078,100024178,77,100024128,100024078,100024178
FC-PFC_vs_FW-PFC:methylDMR.2,5hmC,CG,5hmC|CG_1:10011314-10011406,different,10.927957,3,0.214684,0.369537,0.154853,0.214684,...,PFC,5hmC,CG,1,10011310,10011410,92,10011360,10011310,10011410
FC-PFC_vs_FW-PFC:methylDMR.3,5hmC,CG,5hmC|CG_1:10011416-10011629,different,29.676947,7,0.221502,0.373980,0.152478,0.221502,...,PFC,5hmC,CG,1,10011416,10011629,213,10011522,10011472,10011572
FC-PFC_vs_FW-PFC:methylDMR.4,5hmC,CG,5hmC|CG_1:10011416-10011629,different,29.676947,7,0.221502,0.373980,0.152478,0.221502,...,PFC,5hmC,CG,1,10011416,10011629,213,10011522,10011472,10011572
FC-PFC_vs_FW-PFC:methylDMR.5,5hmC,CG,5hmC|CG_1:10011416-10011629,different,29.676947,7,0.221502,0.373980,0.152478,0.221502,...,PFC,5hmC,CG,1,10011416,10011629,213,10011522,10011472,10011572
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297112,5hmC,CN,5hmC|CN_Y:90805215-90805218,different,29.650025,3,0.185518,0.569000,0.435400,0.133600,...,PFC,5hmC,CN,Y,90805166,90805266,3,90805216,90805166,90805266
297117,5hmC,CN,5hmC|CN_Y:90807478-90807486,different,40.467731,3,0.201060,0.888900,0.688800,0.200100,...,PFC,5hmC,CN,Y,90807432,90807532,8,90807482,90807432,90807532
297118,5hmC,CN,5hmC|CN_Y:90807478-90807486,different,40.467731,3,0.201060,0.888900,0.688800,0.200100,...,PFC,5hmC,CN,Y,90807432,90807532,8,90807482,90807432,90807532
297123,5hmC,CN,5hmC|CN_Y:90807641-90807643,different,25.059346,1,0.190533,0.890000,0.709900,0.180100,...,PFC,5hmC,CN,Y,90807592,90807692,2,90807642,90807592,90807692


In [67]:
for methtype in ['5mC', '5hmC']:
    methtypeL = methtype.lower()
    adata = sc.read_h5ad(f"output/atac1112/3REGIONS_{methtypeL}_new.h5ads")
    dups = adata.var_names.duplicated()
    # Drop duplicated genes
    dmr_mat = adata[:, ~dups].copy()
    #dmr_mat = dmr_mat.obs[~((dmr_mat.obs['celltype.L1_ct']=='OPC') & (dmr_mat.obs['Neurotransmitter_celltype']!='NN'))]
    dmr_mat = dmr_mat[~((dmr_mat.obs['celltype.L1_ct']=='OPC') & (dmr_mat.obs['Neurotransmitter_celltype']!='NN'))]
    adata_dict={}
    for region in ['AMY', 'HIP', 'PFC']:
        df_anno = annodict[methtype]
        df_region = df_anno[df_anno['Region']==region]
        df_region['dmr_name'] = 'chr'+df_region['chr'].astype(str)+':'+df_region['start'].astype(str)+'-'+df_region['end'].astype(str)
        adata_region = dmr_mat[dmr_mat.obs['Region']==region,]
        df_dmr_ano = df_region.drop_duplicates(subset=['dmr_name'])
        df_dmr_ano.drop('gene',axis=1,inplace=True)
        df_dmr_ano.set_index('dmr_name', inplace=True)
        adata_region = adata_region[:, df_dmr_ano.index]
        adata_region.var= df_dmr_ano.loc[adata_region.var_names, :]
        adata_region.var['chr'] = "chr"+adata_region.var['chr'].astype(str)
        #print(region, df_region.shape[0])
        adata_region.write_h5ad(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}.h5ad")


... storing 'mod' as categorical
... storing 'motif' as categorical
... storing 'ifdifferent' as categorical
... storing 'comparision' as categorical
... storing 'tools' as categorical
... storing 'anno' as categorical
... storing 'Region' as categorical
... storing 'methtype' as categorical
... storing 'motif2' as categorical
... storing 'chr' as categorical
... storing 'mod' as categorical
... storing 'motif' as categorical
... storing 'ifdifferent' as categorical
... storing 'comparision' as categorical
... storing 'tools' as categorical
... storing 'anno' as categorical
... storing 'Region' as categorical
... storing 'methtype' as categorical
... storing 'motif2' as categorical
... storing 'chr' as categorical
... storing 'mod' as categorical
... storing 'motif' as categorical
... storing 'ifdifferent' as categorical
... storing 'comparision' as categorical
... storing 'tools' as categorical
... storing 'anno' as categorical
... storing 'Region' as categorical
... storing 'methtype

In [14]:
def g_test_row(row):
    row = np.array(row, dtype=float)
    total = row.sum()
    expected = np.repeat(total/len(row), len(row))

    # Avoid log(0)
    row_safe = np.where(row > 0, row, 1e-12)

    G = 2 * np.sum(row_safe * np.log(row_safe / expected))
    pval = 1 - chi2.cdf(G, df=len(row)-1)
    return pval

for methtype in ['5mC', '5hmC']:
    for region in ['AMY', 'HIP', 'PFC']:
        adata_region = sc.read_h5ad(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}.h5ad")
        adata_region.layers['counts'] = adata_region.X
        adata_region.X = adata_region.layers['counts']
        sc.pp.normalize_total(adata_region, target_sum=1e6)
        #sc.pp.log1p(adata_region)
        group_key = "celltype.L1_ct"  # 替换为 obs 的列名，例如 "cell_type"
        agg_adata = sc.get.aggregate(adata_region,by=group_key,func='mean')
        agg_adata.X = agg_adata.layers['mean']
        # # softmax of over 9 classes
        # from scipy.special import softmax
        X = agg_adata.X.T
        X_norm = X / X.sum(axis=1, keepdims=True)
        X_norm = np.nan_to_num(X_norm, nan=1/9)
        df_xnorm = pd.DataFrame(X_norm, index=agg_adata.var_names, columns=agg_adata.obs_names)
        df_xnorm_safe = df_xnorm.replace(0, 1e-12)
        entropy_values = -np.sum(df_xnorm_safe * np.log(df_xnorm_safe), axis=1)
        # 放回数据框
        df_xnorm['entropy'] = entropy_values
        # pvals = np.array([g_test_row(row) for row in agg_adata.X.T])
        # df_xnorm['pval'] = pvals
        df_xnorm.to_csv(f"/data2st1/junyi/output/atac1112/subset/dmr_region_nt/{region}_{methtype}_celltype_fraction.csv")
        #
        sc.pp.log1p(adata_region)
        sc.tl.rank_genes_groups(adata_region, groupby=group_key, method="wilcoxon",pts=True)
        for ct in adata_region.obs[group_key].unique():
            df_cts = sc.get.rank_genes_groups_df(adata_region, group=ct,pval_cutoff=0.05)
            df_cts.to_csv(f'/data2st1/junyi/output/atac1112/dar/cts/dmr_wilcoxon/{region}_{methtype}_wilcox_{ct}.csv', index=False)
